# 🚀 GRU - COLAB (ListOps Only)

**Bu notebook sadece ListOps çalıştırır.**

HPC'de paralel olarak CIFAR-10 + Selective Copying çalışacak.

## Tahmini Süre: ~4-5 saat (A100/T4)

### ListOps Config (20 epochs)
| Comb | d_model | n_layers | lr | Mamba Acc |
|------|---------|----------|-------|----------|
| 1 | 64 | 4 | 0.001 | 0.49 |
| 2 | 128 | 2 | 0.0005 | 0.30 |

In [ ]:
import time
notebook_start_time = time.time()

In [ ]:
!pip install -q uv
!uv pip install torch numpy lightning huggingface_hub pandas tqdm --quiet

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import lightning as L
from huggingface_hub import hf_hub_download
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader, Dataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
NUM_WORKERS = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ListOps Config - FROM PRESENTATION
LISTOPS_CONFIG = {
    "combination_1": {"d_model": 64,  "n_layers": 4, "lr": 1e-3},
    "combination_2": {"d_model": 128, "n_layers": 2, "lr": 5e-4},
}
LISTOPS_BATCH_SIZES = {"combination_1": 8, "combination_2": 4}
LISTOPS_EPOCHS = 20

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


class GRUBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.gru = nn.GRU(d_model, d_model, 1, batch_first=True, bidirectional=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        normed = self.norm(x)
        output, _ = self.gru(normed)
        return x + self.dropout(output)


class GRUClassifier(L.LightningModule):
    def __init__(self, vocab_size, d_model=128, n_layers=2, num_classes=10, lr=5e-4, weight_decay=0.01, dropout=0.1):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.weight_decay = weight_decay

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([GRUBlock(d_model, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)
        return self.classifier(x)

    def _shared_step(self, batch, stage):
        logits = self(batch['input_ids'], batch.get('attention_mask'))
        loss = F.cross_entropy(logits, batch['labels'])
        acc = (logits.argmax(dim=-1) == batch['labels']).float().mean()
        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_acc', acc, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx): return self._shared_step(batch, 'train')
    def validation_step(self, batch, batch_idx): return self._shared_step(batch, 'val')
    def test_step(self, batch, batch_idx): return self._shared_step(batch, 'test')

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)

In [ ]:
class ListOpsDataset(Dataset):
    def __init__(self, data, max_len=2048):
        self.data = data
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len
        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# Download ListOps
print("Downloading ListOps...")
hf_hub_download(repo_id="monteirot/lra-benchmarks", filename="listops.pt", repo_type="dataset", local_dir=".")
data_listops = torch.load('listops.pt', weights_only=False)
print(f"Train: {len(data_listops['train'].data):,} | Val: {len(data_listops['val'].data):,} | Test: {len(data_listops['test'].data):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train: 96,000 | Val: 2,000 | Test: 2,000


In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint

def make_trainer(num_epochs, name):
    checkpoint_callback = ModelCheckpoint(
        dirpath=f"checkpoints/{name}",
        filename="{epoch}-{val_loss:.2f}",
        save_top_k=1,
        monitor="val_loss",
        mode="min",
        save_last=True
    )

    return L.Trainer(
        max_epochs=num_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed' if torch.cuda.is_available() else 32,
        logger=CSVLogger(save_dir="logs", name=name),
        callbacks=[checkpoint_callback],
    )


In [ ]:
results_listops = {}
num_classes = max([item[1] for item in data_listops['train'].data]) + 1

for combo_name, cfg in LISTOPS_CONFIG.items():
    bs = LISTOPS_BATCH_SIZES[combo_name]
    print(f"\n{'='*60}")
    print(f"  ListOps — {combo_name}: d={cfg['d_model']}, L={cfg['n_layers']}, lr={cfg['lr']}, bs={bs}")
    print(f"{'='*60}")

    train_loader = DataLoader(ListOpsDataset(data_listops['train'].data), batch_size=bs, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(ListOpsDataset(data_listops['val'].data), batch_size=bs, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(ListOpsDataset(data_listops['test'].data), batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    model = GRUClassifier(vocab_size=data_listops['vocab_size'], d_model=cfg['d_model'], n_layers=cfg['n_layers'], num_classes=num_classes, lr=cfg['lr'])
    print(f"  Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    trainer = make_trainer(LISTOPS_EPOCHS, f"listops-{combo_name}")
    import os
    ckpt_path = f"checkpoints/listops-{combo_name}/last.ckpt"
    if os.path.exists(ckpt_path):
        print(f"Resuming from checkpoint: {ckpt_path}")
        trainer.fit(model, train_loader, val_loader, ckpt_path=ckpt_path)
    else:
        trainer.fit(model, train_loader, val_loader)
    trainer.test(model, test_loader)

    test_acc = trainer.callback_metrics.get("test_acc")
    if isinstance(test_acc, torch.Tensor): test_acc = test_acc.item()

    results_listops[combo_name] = {"config": cfg, "test_acc": test_acc, "params": sum(p.numel() for p in model.parameters())}
    print(f"\n  ✓ {combo_name}: test_acc = {test_acc:.4f}")

Testing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63/63 0:00:00 • 0:00:00 109.18it/s

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.476500004529953     │
│         test_loss         │    1.3695051670074463     │
└───────────────────────────┴───────────────────────────┘


  ✓ combination_2: test_acc = 0.4765


In [ ]:
print("\n" + "="*60)
print("  LISTOPS RESULTS (COLAB)")
print("="*60)
print(f"\n  {'Comb':<15} {'Mamba':>10} {'GRU':>10} {'Diff':>10}")
print(f"  {'-'*50}")

mamba_listops = {"combination_1": 0.49, "combination_2": 0.30}

for combo in ["combination_1", "combination_2"]:
    m = mamba_listops[combo]
    g = results_listops[combo]["test_acc"]
    diff = g - m
    print(f"  {combo:<15} {m:>10.4f} {g:>10.4f} {diff:>+10.4f}")

# Save for merging later
torch.save(results_listops, "gru_listops_results.pt")
print("\n✓ Saved to gru_listops_results.pt")


  LISTOPS RESULTS (COLAB)

  Comb                 Mamba        GRU       Diff
  --------------------------------------------------
  combination_1       0.4900     0.2940    -0.1960
  combination_2       0.3000     0.5160    +0.2160

✓ Saved to gru_listops_results.pt


In [ ]:
total_time = time.time() - notebook_start_time
print(f"\n⏱️ Total time: {int(total_time//3600)}h {int((total_time%3600)//60)}m {int(total_time%60)}s")


⏱️ Total time: 7h 49m 8s
